In [ ]:
# Setup, Duplikat-Check & Datumsbereich
# Genutzt für Daten in Kapitel 3.1 (Datensatzbeschreibung, Duplikate, Zeiträume)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import to_rgb
from pathlib import Path

UNI_GREEN = "#009260"  

def mix_with_white(hex_color: str, t: float):
    """
    t in [0,1]. 0 = original color, 1 = white.
    Returns an RGB tuple usable by matplotlib.
    """
    r, g, b = to_rgb(hex_color)
    r = r + (1 - r) * t
    g = g + (1 - g) * t
    b = b + (1 - b) * t
    return (r, g, b)


GREEN_SHADES = [
    mix_with_white(UNI_GREEN, 0.70),
    mix_with_white(UNI_GREEN, 0.55),
    mix_with_white(UNI_GREEN, 0.40),
    mix_with_white(UNI_GREEN, 0.25),
    mix_with_white(UNI_GREEN, 0.10),
]

DATA_PATH = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews.csv"

OUT_DIR = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"File not found: {DATA_PATH}")


df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

dup_full = df.duplicated().sum()

dup_brand_id = None
dup_id_only = None

if {"Brand", "ID"}.issubset(df.columns):
    dup_brand_id = df.duplicated(subset=["Brand", "ID"]).sum()

if "ID" in df.columns:
    dup_id_only = df.duplicated(subset=["ID"]).sum()

print("=== DUPLICATE CHECK ===")
print(f"Rows total: {len(df):,}")
print(f"Duplicate full rows: {dup_full:,}")
if dup_brand_id is not None:
    print(f"Duplicate by (Brand, ID): {dup_brand_id:,}")
if dup_id_only is not None:
    print(f"Duplicate by (ID): {dup_id_only:,}")

if dup_full > 0:
    df[df.duplicated()].to_csv(OUT_DIR / "duplicates_full_rows.csv", index=False)
if dup_brand_id is not None and dup_brand_id > 0:
    df[df.duplicated(subset=["Brand", "ID"])].to_csv(OUT_DIR / "duplicates_brand_id.csv", index=False)
if dup_id_only is not None and dup_id_only > 0:
    df[df.duplicated(subset=["ID"])].to_csv(OUT_DIR / "duplicates_id.csv", index=False)


DATE_COL = "Date"
if DATE_COL not in df.columns:
    raise KeyError(f"Expected column '{DATE_COL}' not found. Columns are: {df.columns.tolist()}")

df["date_dt"] = pd.to_datetime(df[DATE_COL], errors="coerce")

date_ranges = (
    df.groupby("Brand")["date_dt"]
      .agg(min_date="min", max_date="max", n_with_date="count")
      .reset_index()
)

print("\n=== DATE RANGE BY BRAND (from dataset 'Date' column) ===")
print(date_ranges.to_string(index=False))

date_ranges.to_csv(OUT_DIR / "date_range_by_brand.csv", index=False)
print("Saved:", OUT_DIR / "date_range_by_brand.csv")


RATING_COL = "Rating"
if RATING_COL not in df.columns:
    raise KeyError(f"Expected column '{RATING_COL}' not found. Columns are: {df.columns.tolist()}")

df["rating_num"] = pd.to_numeric(df[RATING_COL], errors="coerce")
df_r = df.dropna(subset=["rating_num"]).copy()
df_r = df_r[df_r["rating_num"].between(1, 5)].copy()
df_r["rating"] = df_r["rating_num"].round().astype(int)

def rating_dist_counts_pct(sub: pd.DataFrame):
    counts = sub["rating"].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0).astype(int)
    pct = (counts / len(sub) * 100).round(2)
    return counts, pct

counts, pct = rating_dist_counts_pct(df_r)
mean_rating = df_r["rating_num"].mean()

dist_table = pd.DataFrame({"stars": [1, 2, 3, 4, 5], "count": counts.values, "pct": pct.values})
dist_table.to_csv(OUT_DIR / "star_distribution_overall.csv", index=False)

print("\n=== STAR DISTRIBUTION (OVERALL) ===")
print(dist_table.to_string(index=False))
print("Saved:", OUT_DIR / "star_distribution_overall.csv")


fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111)

x = np.array([1, 2, 3, 4, 5]).astype(str)
y = pct.values

ax.bar(x, y, color=GREEN_SHADES, edgecolor=UNI_GREEN, linewidth=0.8)

ax.set_title(f"Overall Star Rating Distribution (n={len(df_r):,})\nMean rating = {mean_rating:.2f}")
ax.set_xlabel("Star rating")
ax.set_ylabel("Share of reviews (%)")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))

for i, v in enumerate(y):
    ax.text(i, v, f"{v:.2f}%", ha="center", va="bottom", color=UNI_GREEN)

fig.tight_layout()
fig.savefig(OUT_DIR / "stars_overall_distribution_uni_green.png", dpi=250)
plt.close(fig)

print("\nSaved figure:", OUT_DIR / "stars_overall_distribution_uni_green.png")


CATEGORICAL_COLS = ["Brand", "Gender", "Sustainability", "Incentivized", "Locale"]

rows = []
total = len(df)
for col in CATEGORICAL_COLS:
    if col not in df.columns:
        print(f"[SKIP] Column not found: {col}")
        continue

    vc = df[col].value_counts(dropna=False)
    for val, c in vc.items():
        rows.append({
            "variable": col,
            "value": val,
            "count": int(c),
            "pct": round(c / total * 100, 2)
        })

cat_table = pd.DataFrame(rows)
cat_table.to_csv(OUT_DIR / "category_counts_overall.csv", index=False)

print("\nSaved category counts:", OUT_DIR / "category_counts_overall.csv")


In [ ]:
# Genutzt zum Erstellen von Fig. 6: Overall Star Rating Distribution (n=16,243)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import to_rgb
from matplotlib.ticker import PercentFormatter

DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews.csv"
OUT  = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT.mkdir(parents=True, exist_ok=True)

UNI_GREEN = "#009260"
def mix(hex_color, t):  
    r,g,b = to_rgb(hex_color)
    return (r+(1-r)*t, g+(1-g)*t, b+(1-b)*t)
SHADES = [mix(UNI_GREEN,t) for t in [0.82,0.72,0.60,0.35,0.00]]  


df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()

print("Duplicates | full rows:", df.duplicated().sum(),
      "| (Brand,ID):", df.duplicated(["Brand","ID"]).sum() if {"Brand","ID"}.issubset(df.columns) else "n/a",
      "| ID:", df.duplicated(["ID"]).sum() if "ID" in df.columns else "n/a")


if {"Brand","Date"}.issubset(df.columns):
    d = pd.to_datetime(df["Date"], errors="coerce")
    print("\nDate range by brand (from 'Date' column):")
    print(pd.DataFrame({"min": d.groupby(df["Brand"]).min(),
                        "max": d.groupby(df["Brand"]).max(),
                        "n_with_date": d.groupby(df["Brand"]).count()}))

for c in ["Brand","Gender","Sustainability","Incentivized","Locale"]:
    if c in df.columns:
        print(f"\n{c} counts:\n{df[c].value_counts(dropna=False)}")


r = pd.to_numeric(df["Rating"], errors="coerce")
r = r[r.between(1,5)]
stars = r.round().astype(int)

counts = stars.value_counts().reindex([1,2,3,4,5], fill_value=0)
pct = (counts / len(stars) * 100).round(1)
mean = r.mean()
ymax = max(80, int(np.ceil(pct.max()/10)*10))

fig, ax = plt.subplots(figsize=(9, 5.2))
x = np.arange(5)
bars = ax.bar(x, pct.values, color=SHADES, width=0.8)

ax.set_xticks(x, [f"{i}★" for i in range(1,6)])
ax.set_ylim(0, ymax)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel("Star rating")
ax.set_ylabel("Share of reviews")
ax.set_title(f"Overall Star Rating Distribution (n={len(stars):,})", fontsize=18)
ax.text(0.5, 0.86, f"Ø Mean = {mean:.2f}", transform=ax.transAxes,
        ha="center", color=UNI_GREEN, fontsize=14, fontweight="bold")

ax.grid(axis="y", linestyle="--", alpha=0.18)
ax.grid(axis="x", linestyle=":",  alpha=0.12)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#cccccc")
ax.spines["bottom"].set_color("#cccccc")

for b, v in zip(bars, pct.values):
    ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}%", ha="center", fontsize=11)

fig.tight_layout()
fig.savefig(OUT / "stars_overall_distribution_uni_green.png", dpi=300)
plt.close(fig)
print("\nSaved:", OUT / "stars_overall_distribution_uni_green.png")


In [ ]:
# Genutzt zum Erstellen von Tabelle 3: Overview of the dataset

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path


DATA    = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews.csv"
OUT_DIR = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()


df["rating_num"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["rating_num"].between(1, 5)].copy()
df["rating"] = df["rating_num"].round().astype(int)

if "Date" in df.columns:
    df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
else:
    df["date"] = pd.NaT

def _span(sub):
    a, b = sub["date"].min(), sub["date"].max()
    return "" if pd.isna(a) or pd.isna(b) else f"{a:%d/%m/%Y}–{b:%d/%m/%Y}"

def _row(sub, label, indent=False):
    lab = ("   " + label) if indent else label
    n = len(sub)
    mean = sub["rating_num"].mean()
    s45 = sub["rating"].isin([4, 5]).mean() * 100
    s12 = sub["rating"].isin([1, 2]).mean() * 100
    return [lab, f"{n:,}", f"Ø{mean:.2f}", f"{s45:.2f}", f"{s12:.2f}", _span(sub)]


cols = ["Group", "n", "Mean ★-rating", "4–5★ in %", "1–2★ in %", "Time span"]
rows = []
brand_header_rows = []

rows.append(_row(df, "Overall"))

if "Brand" in df.columns:
    for brand in ["Adidas", "Nike"]:
        if brand not in set(df["Brand"].dropna().unique()):
            continue

        bdf = df[df["Brand"] == brand].copy()

        brand_header_rows.append(len(rows) + 1)   
        rows.append([brand, "", "", "", "", ""]) 

        rows.append(_row(bdf, "All", indent=True))

        if "Sustainability" in bdf.columns:
            for subg in ["Sustainable", "Conventional"]:
                if subg in set(bdf["Sustainability"].dropna().unique()):
                    rows.append(_row(bdf[bdf["Sustainability"] == subg], subg, indent=True))

table_df = pd.DataFrame(rows, columns=cols)


available_fonts = {f.name for f in font_manager.fontManager.ttflist}
mpl.rcParams.update({"font.family": "Aptos" if "Aptos" in available_fonts else "DejaVu Sans"})


fig_w = 10.5

fig_h = 0.60 + 0.40 * (len(table_df) + 1) 

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

col_widths = [0.33, 0.10, 0.16, 0.16, 0.16, 0.25]

tbl = ax.table(
    cellText=table_df.values.tolist(),
    colLabels=cols,
    cellLoc="center",
    colLoc="center",
    colWidths=col_widths,
    loc="center",
)

tbl.auto_set_font_size(False)
body_fs, header_fs = 12.0, 12.0

for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_linewidth(0.0)
    cell.set_facecolor("white")

 
    cell.set_height(0.19 if r == 0 else 0.17)  

    if r == 0:
        cell.get_text().set_weight("bold")
        cell.get_text().set_fontsize(header_fs)
    else:
        cell.get_text().set_fontsize(body_fs)

    if c == 0:
        cell.get_text().set_ha("left")
        cell.PAD = 0.02


for rr in brand_header_rows:
    for c in range(len(cols)):
        cell = tbl[rr, c]
        cell.set_facecolor("#f2f2f2")
        cell.get_text().set_weight("bold")
        if c != 0:
            cell.get_text().set_text("")


fig.canvas.draw()
cells = tbl.get_celld()

header_cell = cells[(0, 0)]
y_header_bottom = header_cell.get_y()
y_header_top = y_header_bottom + header_cell.get_height()

last_r = len(table_df)
bottom_cell = cells[(last_r, 0)]
y_table_bottom = bottom_cell.get_y()

x_left = header_cell.get_x()
x_right = cells[(0, len(cols) - 1)].get_x() + cells[(0, len(cols) - 1)].get_width()

ax.plot([x_left, x_right], [y_header_top, y_header_top], color="black", lw=1.2, transform=ax.transAxes, clip_on=False)
ax.plot([x_left, x_right], [y_header_bottom, y_header_bottom], color="black", lw=0.9, transform=ax.transAxes, clip_on=False)
ax.plot([x_left, x_right], [y_table_bottom, y_table_bottom], color="black", lw=1.2, transform=ax.transAxes, clip_on=False)

for rr in brand_header_rows:
    y = cells[(rr, 0)].get_y() + cells[(rr, 0)].get_height()
    ax.plot([x_left, x_right], [y, y], color="black", lw=0.8, transform=ax.transAxes, clip_on=False)

out_path = OUT_DIR / "overview_table_booktabs.png"
fig.tight_layout(pad=0.15)
fig.savefig(out_path, dpi=300, bbox_inches="tight", transparent=False)
plt.close(fig)

print("Saved:", out_path)


In [ ]:
# Incentivized vs. Organic: Welch T-Test & Cohen's d
# Genutzt für Daten in Kapitel 3.1 (Incentivized-Analyse)
# Erstellt außerdem die gefilterte Datei All_Reviews_Organic

from pathlib import Path
import pandas as pd
import numpy as np
import math
from scipy.stats import ttest_ind, t as t_dist


DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews.csv"

print("Using:", DATA)
print("Exists:", DATA.exists())

df = pd.read_csv(DATA)


d = df.copy()

d["Rating"] = pd.to_numeric(d["Rating"], errors="coerce")

d["Incentivized"] = d["Incentivized"].map({"True": True, "False": False, True: True, False: False})

d = d.dropna(subset=["Rating", "Incentivized"]).copy()


x = d.loc[d["Incentivized"] == True, "Rating"].astype(float)   
y = d.loc[d["Incentivized"] == False, "Rating"].astype(float)  


n_x, n_y = len(x), len(y)
mx, my = x.mean(), y.mean()
sx, sy = x.std(ddof=1), y.std(ddof=1)
diff = mx - my

print("\n--- Descriptives ---")
print(f"Incentivized:     n={n_x}, mean={mx:.3f}, sd={sx:.3f}")
print(f"Not incentivized: n={n_y}, mean={my:.3f}, sd={sy:.3f}")
print(f"Mean diff (incent - not) = {diff:.3f}")


res = ttest_ind(x, y, equal_var=False, alternative="two-sided")


se = math.sqrt(sx**2 / n_x + sy**2 / n_y)
df_w = (sx**2 / n_x + sy**2 / n_y) ** 2 / (
    (sx**2 / n_x) ** 2 / (n_x - 1) + (sy**2 / n_y) ** 2 / (n_y - 1)
)

print("\n--- Welch t-test ---")
print(f"Welch t = {res.statistic:.3f}, p = {res.pvalue:.3e}, df = {df_w:.1f}")


tcrit = t_dist.ppf(0.975, df_w)
ci_low, ci_high = diff - tcrit * se, diff + tcrit * se
print(f"95% CI for mean diff: [{ci_low:.3f}, {ci_high:.3f}]")


sp = math.sqrt(((n_x - 1) * sx**2 + (n_y - 1) * sy**2) / (n_x + n_y - 2))
cohen_d = diff / sp
print(f"Cohen's d = {cohen_d:.3f}")


OUT = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_no_incentivized.csv"
df_no_incent = df[df["Incentivized"].map({"True": True, "False": False, True: True, False: False}) == False].copy()
df_no_incent.to_csv(OUT, index=False)

print("\nSaved:", OUT)


In [ ]:
# Incentivized vs. Organic: Übersichtstabelle
# Genutzt als Ergänzung für Kapitel 3.1 (Incentivized-Statistiken pro Brand)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path


BASE = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Datensatz"
ADIDAS_PATH = BASE / "Adidas" / "Adidas_Total.csv"
NIKE_PATH   = BASE / "Nike" / "Nike_Total.csv"

OUT_DIR = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [ADIDAS_PATH, NIKE_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")


adf = pd.read_csv(ADIDAS_PATH); adf["Brand"] = "Adidas"
ndf = pd.read_csv(NIKE_PATH);   ndf["Brand"] = "Nike"
df = pd.concat([adf, ndf], ignore_index=True)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"]).copy()
df["rating"] = df["rating"].astype(int)
df = df[df["rating"].between(1, 5)].copy()


df["incentivized_flag"] = pd.NA
if "incentivized_review" in df.columns:
    df.loc[df["Brand"]=="Adidas", "incentivized_flag"] = df.loc[df["Brand"]=="Adidas", "incentivized_review"]
if "incentivized" in df.columns:
    df.loc[df["Brand"]=="Nike", "incentivized_flag"] = df.loc[df["Brand"]=="Nike", "incentivized"]
df["incentivized_flag"] = df["incentivized_flag"].astype("boolean")

df["Review Type"] = np.where(df["incentivized_flag"] == True, "Incentivized", "Organic")


table1 = (
    df.groupby(["Brand", "Review Type"])
      .agg(
          N=("rating", "size"),
          Mean=("rating", "mean"),
          share_5=("rating", lambda x: (x == 5).mean() * 100),
          share_1_2=("rating", lambda x: (x <= 2).mean() * 100),
      )
      .reset_index()
)

table1["Share within Brand (%)"] = table1["N"] / table1.groupby("Brand")["N"].transform("sum") * 100


table1["N"] = table1["N"].astype(int).map(lambda x: f"{x:,}")
table1["Share within Brand (%)"] = table1["Share within Brand (%)"].map(lambda x: f"{x:.2f}")
table1["Mean"] = table1["Mean"].map(lambda x: f"Ø{x:.2f}")
table1["5★ share (%)"] = table1["share_5"].map(lambda x: f"{x:.2f}")
table1["1–2★ share (%)"] = table1["share_1_2"].map(lambda x: f"{x:.2f}")
table1 = table1.drop(columns=["share_5", "share_1_2"])


order_type = pd.CategoricalDtype(["Organic", "Incentivized"], ordered=True)
table1["Review Type"] = table1["Review Type"].astype(order_type)
table1 = table1.sort_values(["Brand", "Review Type"]).reset_index(drop=True)

table1 = table1.rename(columns={
    "Share within Brand (%)": "Share within\nbrand (%)",
    "5★ share (%)": "5★ share\n(%)",
    "1–2★ share (%)": "1–2★ share\n(%)",
})

table1 = table1[[
    "Brand", "Review Type", "N", "Share within\nbrand (%)",
    "Mean", "5★ share\n(%)", "1–2★ share\n(%)"
]]


available_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT_FAMILY = "Aptos" if "Aptos" in available_fonts else "DejaVu Sans"
mpl.rcParams.update({"font.family": FONT_FAMILY})


fig_w = 7.4   
fig_h = 2.1
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

col_labels = list(table1.columns)
cell_text = table1.values.tolist()


col_widths = [0.09, 0.16, 0.09, 0.23, 0.09, 0.17, 0.17]

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc="center",
    colLoc="center",
    colWidths=col_widths,
    loc="center"
)

tbl.auto_set_font_size(False)

body_fs = 10.5
header_fs = 10.0  

for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor("white")
    cell.set_edgecolor("white")
    cell.set_linewidth(0.0)
    cell.set_height(0.23 if r == 0 else 0.20)
    if r == 0:
        cell.get_text().set_weight("bold")
        cell.get_text().set_fontsize(header_fs)
    else:
        cell.get_text().set_fontsize(body_fs)


fig.canvas.draw()
cells = tbl.get_celld()

header_cell = cells[(0, 0)]
y_header_bottom = header_cell.get_y()
y_header_top = y_header_bottom + header_cell.get_height()

last_row = len(cell_text)
bottom_cell = cells[(last_row, 0)]
y_table_bottom = bottom_cell.get_y()

x_left = header_cell.get_x()
last_header_cell = cells[(0, len(col_labels) - 1)]
x_right = last_header_cell.get_x() + last_header_cell.get_width()

ax.plot([x_left, x_right], [y_header_top, y_header_top], color="black", linewidth=1.1, transform=ax.transAxes, clip_on=False)
ax.plot([x_left, x_right], [y_header_bottom, y_header_bottom], color="black", linewidth=0.9, transform=ax.transAxes, clip_on=False)
ax.plot([x_left, x_right], [y_table_bottom, y_table_bottom], color="black", linewidth=1.1, transform=ax.transAxes, clip_on=False)

out_path = OUT_DIR / "Table1_basic_booktabs_A4_no_overlap.png"
fig.tight_layout(pad=0.2)
fig.savefig(out_path, dpi=300, bbox_inches="tight", transparent=False)
plt.close(fig)

print("Saved:", out_path)


In [ ]:

# Genutzt zum Erstellen von Tabelle 4:Overview of the cleaned dataset (Organic, n=8,298)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path


DATA    = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_Organic.csv"
OUT_DIR = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()


df["rating_num"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["rating_num"].between(1, 5)].copy()
df["rating"] = df["rating_num"].round().astype(int)

if "Date" in df.columns:
    df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
else:
    df["date"] = pd.NaT

def _span(sub):
    a, b = sub["date"].min(), sub["date"].max()
    return "" if pd.isna(a) or pd.isna(b) else f"{a:%d/%m/%Y}–{b:%d/%m/%Y}"

def _row(sub, label, indent=False):
    lab = ("   " + label) if indent else label
    n = len(sub)
    mean = sub["rating_num"].mean()
    s45 = sub["rating"].isin([4, 5]).mean() * 100
    s12 = sub["rating"].isin([1, 2]).mean() * 100
    return [lab, f"{n:,}", f"Ø{mean:.2f}", f"{s45:.2f}", f"{s12:.2f}", _span(sub)]


cols = ["Group", "n", "Mean ★-rating", "4–5★ in %", "1–2★ in %", "Time span"]
rows = []
brand_header_rows = []

rows.append(_row(df, "Overall"))

if "Brand" in df.columns:
    for brand in ["Adidas", "Nike"]:
        if brand not in set(df["Brand"].dropna().unique()):
            continue

        bdf = df[df["Brand"] == brand].copy()

        brand_header_rows.append(len(rows) + 1)   
        rows.append([brand, "", "", "", "", ""]) 

        rows.append(_row(bdf, "All", indent=True))

        if "Sustainability" in bdf.columns:
            for subg in ["Sustainable", "Conventional"]:
                if subg in set(bdf["Sustainability"].dropna().unique()):
                    rows.append(_row(bdf[bdf["Sustainability"] == subg], subg, indent=True))

table_df = pd.DataFrame(rows, columns=cols)


available_fonts = {f.name for f in font_manager.fontManager.ttflist}
mpl.rcParams.update({"font.family": "Aptos" if "Aptos" in available_fonts else "DejaVu Sans"})


fig_w = 10.5

fig_h = 0.60 + 0.40 * (len(table_df) + 1) 

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

col_widths = [0.33, 0.10, 0.16, 0.16, 0.16, 0.25]

tbl = ax.table(
    cellText=table_df.values.tolist(),
    colLabels=cols,
    cellLoc="center",
    colLoc="center",
    colWidths=col_widths,
    loc="center",
)

tbl.auto_set_font_size(False)
body_fs, header_fs = 12.0, 12.0

for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_linewidth(0.0)
    cell.set_facecolor("white")

  
    cell.set_height(0.19 if r == 0 else 0.17)  

    if r == 0:
        cell.get_text().set_weight("bold")
        cell.get_text().set_fontsize(header_fs)
    else:
        cell.get_text().set_fontsize(body_fs)

    if c == 0:
        cell.get_text().set_ha("left")
        cell.PAD = 0.02


for rr in brand_header_rows:
    for c in range(len(cols)):
        cell = tbl[rr, c]
        cell.set_facecolor("#f2f2f2")
        cell.get_text().set_weight("bold")
        if c != 0:
            cell.get_text().set_text("")

fig.canvas.draw()
cells = tbl.get_celld()

header_cell = cells[(0, 0)]
y_header_bottom = header_cell.get_y()
y_header_top = y_header_bottom + header_cell.get_height()

last_r = len(table_df)
bottom_cell = cells[(last_r, 0)]
y_table_bottom = bottom_cell.get_y()

x_left = header_cell.get_x()
x_right = cells[(0, len(cols) - 1)].get_x() + cells[(0, len(cols) - 1)].get_width()

ax.plot([x_left, x_right], [y_header_top, y_header_top], color="black", lw=1.2, transform=ax.transAxes, clip_on=False)
ax.plot([x_left, x_right], [y_header_bottom, y_header_bottom], color="black", lw=0.9, transform=ax.transAxes, clip_on=False)
ax.plot([x_left, x_right], [y_table_bottom, y_table_bottom], color="black", lw=1.2, transform=ax.transAxes, clip_on=False)

for rr in brand_header_rows:
    y = cells[(rr, 0)].get_y() + cells[(rr, 0)].get_height()
    ax.plot([x_left, x_right], [y, y], color="black", lw=0.8, transform=ax.transAxes, clip_on=False)

out_path = OUT_DIR / "overview_table_booktabs.png"
fig.tight_layout(pad=0.15)
fig.savefig(out_path, dpi=300, bbox_inches="tight", transparent=False)
plt.close(fig)

print("Saved:", out_path)

In [ ]:
# Genutzt zum Erstellen von Fig. 7: Overall Star Rating Distribution (Organic, n=8,298)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import to_rgb
from matplotlib.ticker import PercentFormatter

DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_Organic.csv"
OUT  = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT.mkdir(parents=True, exist_ok=True)

UNI_GREEN = "#009260"
def mix(hex_color, t):
    r,g,b = to_rgb(hex_color)
    return (r+(1-r)*t, g+(1-g)*t, b+(1-b)*t)
SHADES = [mix(UNI_GREEN,t) for t in [0.82,0.72,0.60,0.35,0.00]]

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()

r = pd.to_numeric(df["Rating"], errors="coerce")
r = r[r.between(1,5)]
stars = r.round().astype(int)

counts = stars.value_counts().reindex([1,2,3,4,5], fill_value=0)
pct = (counts / len(stars) * 100).round(1)
mean = r.mean()
ymax = max(80, int(np.ceil(pct.max()/10)*10))

fig, ax = plt.subplots(figsize=(9, 5.2))
x = np.arange(5)
bars = ax.bar(x, pct.values, color=SHADES, width=0.8)

ax.set_xticks(x, [f"{i}★" for i in range(1,6)])
ax.set_ylim(0, ymax)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel("Star rating")
ax.set_ylabel("Share of reviews")
ax.set_title(f"Overall Star Rating Distribution (n={len(stars):,})", fontsize=18)
ax.text(0.5, 0.86, f"Ø Mean = {mean:.2f}", transform=ax.transAxes,
        ha="center", color=UNI_GREEN, fontsize=14, fontweight="bold")

ax.grid(axis="y", linestyle="--", alpha=0.18)
ax.grid(axis="x", linestyle=":",  alpha=0.12)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#cccccc")
ax.spines["bottom"].set_color("#cccccc")

for b, v in zip(bars, pct.values):
    ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}%", ha="center", fontsize=11)

fig.tight_layout()
png_path = OUT / "stars_overall_distribution_uni_green_ORGANIC.png"
pdf_path = OUT / "stars_overall_distribution_uni_green_ORGANIC.pdf"
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path)
plt.close(fig)

print("Saved:", png_path)
print("Saved:", pdf_path)


In [ ]:
# Genutzt zum Erstellen von Fig. 8: Star Rating Distribution by Brand

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import to_rgb
from matplotlib.ticker import PercentFormatter

DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_Organic.csv"
OUT  = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT.mkdir(parents=True, exist_ok=True)

UNI_GREEN = "#009260"
def mix(hex_color, t):
    r, g, b = to_rgb(hex_color)
    return (r + (1 - r) * t, g + (1 - g) * t, b + (1 - b) * t)

SHADES = [mix(UNI_GREEN, t) for t in [0.82, 0.72, 0.60, 0.35, 0.00]]  # 1★..5★

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["Rating"].between(1, 5)].copy()
df["Star"] = df["Rating"].round().astype(int)

brands = ["Adidas", "Nike"]
counts = pd.crosstab(df["Brand"], df["Star"]).reindex(index=brands, columns=[1,2,3,4,5], fill_value=0)
pct = counts.div(counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5.2))
y = np.arange(len(brands))
left = np.zeros(len(brands))

for i, star in enumerate([1,2,3,4,5]):
    vals = pct[star].values
    ax.barh(y, vals, left=left, color=SHADES[i], height=0.65, label=f"{star}★")
    left += vals

ax.set_xlim(0, 105)
ax.xaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel("Share of reviews")

ax.set_yticks(y, [f"{b} (n={counts.loc[b].sum():,})" for b in brands])

ax.set_title("Star Rating Distribution by Brand", fontsize=14)

ax.grid(axis="x", linestyle="--", alpha=0.18)
ax.grid(axis="y", linestyle=":",  alpha=0.12)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#cccccc")
ax.spines["bottom"].set_color("#cccccc")

topbox = pct[5].values
for yi, v in zip(y, topbox):
    ax.text(101.0, yi, f"5★: {v:.1f}%", ha="left", va="center", fontsize=11)

ax.legend(ncol=5, bbox_to_anchor=(0.5, -0.18), loc="upper center", frameon=False)

fig.tight_layout()
png_path = OUT / "stars_distribution_by_brand_horizontal_uni_green_ORGANIC.png"
pdf_path = OUT / "stars_distribution_by_brand_horizontal_uni_green_ORGANIC.pdf"
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path)
plt.close(fig)

print("Saved:", png_path)
print("Saved:", pdf_path)


In [ ]:
# Genutzt zum Erstellen von Fig. 9: Star Rating Distribution by Product Type

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import to_rgb
from matplotlib.ticker import PercentFormatter

DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_Organic.csv"
OUT  = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT.mkdir(parents=True, exist_ok=True)

UNI_GREEN = "#009260"
def mix(hex_color, t):
    r, g, b = to_rgb(hex_color)
    return (r + (1 - r) * t, g + (1 - g) * t, b + (1 - b) * t)

SHADES = [mix(UNI_GREEN, t) for t in [0.82, 0.72, 0.60, 0.35, 0.00]]  # 1★..5★

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["Rating"].between(1, 5)].copy()
df["Star"] = df["Rating"].round().astype(int)

groups = ["Conventional", "Sustainable"]
counts = pd.crosstab(df["Sustainability"], df["Star"]).reindex(index=groups, columns=[1,2,3,4,5], fill_value=0)
pct = counts.div(counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5.2))
y = np.arange(len(groups))
left = np.zeros(len(groups))

for i, star in enumerate([1,2,3,4,5]):
    vals = pct[star].values
    ax.barh(y, vals, left=left, color=SHADES[i], height=0.65, label=f"{star}★")
    left += vals

ax.set_xlim(0, 105)
ax.xaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel("Share of reviews")

ax.set_yticks(y, [f"{g} (n={counts.loc[g].sum():,})" for g in groups])

ax.set_title("Star Rating Distribution by Product Type", fontsize=14)

ax.grid(axis="x", linestyle="--", alpha=0.18)
ax.grid(axis="y", linestyle=":",  alpha=0.12)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#cccccc")
ax.spines["bottom"].set_color("#cccccc")

topbox = pct[5].values
for yi, v in zip(y, topbox):
    ax.text(101.0, yi, f"5★: {v:.1f}%", ha="left", va="center", fontsize=11)

ax.legend(ncol=5, bbox_to_anchor=(0.5, -0.18), loc="upper center", frameon=False)

fig.tight_layout()
png_path = OUT / "stars_distribution_by_producttype_horizontal_uni_green_ORGANIC.png"
pdf_path = OUT / "stars_distribution_by_producttype_horizontal_uni_green_ORGANIC.pdf"
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path)
plt.close(fig)

print("Saved:", png_path)
print("Saved:", pdf_path)


In [ ]:
# Genutzt zum Erstellen von Tabelle 5: Star-Rating Comparison by Product, Brand & Gender

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib import rcParams

DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_Organic.csv"
OUT  = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["Rating"].between(1,5)].copy()

def summarize(sub):
    n = len(sub)
    mean = sub["Rating"].mean()
    p45 = (sub["Rating"] >= 4).mean() * 100
    p12 = (sub["Rating"] <= 2).mean() * 100
    return n, mean, p45, p12

rows = []
rows.append(("Product type (All)", "", "", "", ""))
rows.append(("  Conventional",) + summarize(df[df["Sustainability"]=="Conventional"]))
rows.append(("  Sustainable",) + summarize(df[df["Sustainability"]=="Sustainable"]))

rows.append(("Brand", "", "", "", ""))
rows.append(("  Sustainable Adidas",) + summarize(df[(df["Sustainability"]=="Sustainable") & (df["Brand"]=="Adidas")]))
rows.append(("  Sustainable Nike",) + summarize(df[(df["Sustainability"]=="Sustainable") & (df["Brand"]=="Nike")]))
rows.append(("  Conventional Adidas",) + summarize(df[(df["Sustainability"]=="Conventional") & (df["Brand"]=="Adidas")]))
rows.append(("  Conventional Nike",) + summarize(df[(df["Sustainability"]=="Conventional") & (df["Brand"]=="Nike")]))

rows.append(("Gender", "", "", "", ""))
rows.append(("  Sustainable Female",) + summarize(df[(df["Sustainability"]=="Sustainable") & (df["Gender"]=="Female")]))
rows.append(("  Sustainable Male",) + summarize(df[(df["Sustainability"]=="Sustainable") & (df["Gender"]=="Male")]))
rows.append(("  Conventional Female",) + summarize(df[(df["Sustainability"]=="Conventional") & (df["Gender"]=="Female")]))
rows.append(("  Conventional Male",) + summarize(df[(df["Sustainability"]=="Conventional") & (df["Gender"]=="Male")]))

tbl = pd.DataFrame(rows, columns=["Group", "n", "Mean ★-rating", "4–5★ in %", "1–2★ in %"])

def fmt_int(x):
    if x=="" or pd.isna(x): return ""
    return f"{int(round(x)):,}".replace(",", ".")

def fmt_dec(x, d=2):
    if x=="" or pd.isna(x): return ""
    return f"{x:.{d}f}".replace(".", ",")

fmt = tbl.copy()
fmt["n"] = fmt["n"].apply(fmt_int)
fmt["Mean ★-rating"] = fmt["Mean ★-rating"].apply(lambda x: fmt_dec(x, 2))
fmt["4–5★ in %"] = fmt["4–5★ in %"].apply(lambda x: fmt_dec(x, 2))
fmt["1–2★ in %"] = fmt["1–2★ in %"].apply(lambda x: fmt_dec(x, 2))

rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]

data = fmt.values.tolist()
cols = fmt.columns.tolist()

fig, ax = plt.subplots(figsize=(12.0, 3.6))
ax.axis("off")

table = ax.table(
    cellText=data,
    colLabels=cols,
    cellLoc="center",
    colLoc="center",
    loc="upper left",
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(10)

col_widths = [0.44, 0.12, 0.16, 0.14, 0.14]
for c, w in enumerate(col_widths):
    for r in range(len(data)+1):
        table[(r, c)].set_width(w)

def is_section_row(r_body_idx):
    return fmt.iloc[r_body_idx, 1] == ""


for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#000000")
    cell.set_linewidth(0.0)
    cell.visible_edges = "horizontal"

for c in range(len(cols)):
    cell = table[(0, c)]
    cell.set_facecolor("#f2f2f2")
    cell.set_text_props(weight="bold")
    cell.set_linewidth(1.2)
    cell.visible_edges = "BT"

for r in range(1, len(data)+1):
    section = is_section_row(r-1)
    for c in range(len(cols)):
        cell = table[(r, c)]

        if c == 0:
            cell._loc = "w"
            cell.PAD = 0.002  
            if section:
                cell.set_text_props(weight="bold")
        else:
            cell._loc = "center"

        if section:
            cell.set_facecolor("#e6e6e6")
            cell.set_linewidth(0.9)
            cell.visible_edges = "BT"
        else:
            cell.set_facecolor("#ffffff")
            cell.set_linewidth(0.4)
            cell.visible_edges = "B"

last_r = len(data)
for c in range(len(cols)):
    cell = table[(last_r, c)]
    cell.set_linewidth(1.2)
    cell.visible_edges = "B"

plt.tight_layout()

png_path = OUT / "table_star_comparisons_extended_no_timespan.png"
pdf_path = OUT / "table_star_comparisons_extended_no_timespan.pdf"
fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.close(fig)

print("Saved:", png_path)
print("Saved:", pdf_path)


In [ ]:
#Informal Language Analyse (Emojis, Slang, Caps, Punctuation)

from pathlib import Path
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager


BASE = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Datensatz"
INPUT_CSV = BASE / "Brand_Total_1.csv"

OUT_DIR = BASE / "Methodology" / "Tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(INPUT_CSV, low_memory=False)

if "title" not in df.columns:
    df["title"] = ""

df["brand"] = df["brand"].astype(str)
df["text"] = df["text"].fillna("").astype(str)
df["title"] = df["title"].fillna("").astype(str)

df["full_text"] = (df["title"].str.strip() + " " + df["text"].str.strip()).str.strip()

TOTAL_N = len(df)


EMOJI_RE = re.compile(
    "[" 
    "\U0001F300-\U0001FAFF"      
    "\U0001F1E6-\U0001F1FF"      
    "\u2600-\u27BF"             
    "]+",
    flags=re.UNICODE
)

EMOTICON_RE = re.compile(r"(:-\)|:\)|:-\(|:\(|;-\)|;\)|:D|:-D|xD|XD|:\'\(|:\||:-\||<3)")

SLANG_ABBREV = [
    "lol","lmao","rofl","omg","wtf","idk","imo","imho","tbh","brb","smh","lmk",
    "rn","ngl","btw","fyi","jk","afaik","asap","yolo"
]
SLANG_ABBREV_RE = re.compile(r"\b(?:%s)\b" % "|".join(map(re.escape, SLANG_ABBREV)), flags=re.IGNORECASE)

COLLOQUIAL_WORDS = ["comfy","sick","dope","lit","kinda","gonna","wanna","tho","cuz","pls","plz","luv","meh","bruh"]
COLLOQUIAL_RE = re.compile(r"\b(?:%s)\b" % "|".join(map(re.escape, COLLOQUIAL_WORDS)), flags=re.IGNORECASE)

EXCL_3_RE = re.compile(r"!{3,}")    
Q_3_RE    = re.compile(r"\?{3,}")

ALLCAPS_WORD_RE = re.compile(r"\b[A-Z]{3,}\b")

ELONGATED_RE = re.compile(r"(.)\1{2,}", flags=re.IGNORECASE)

def has(pat, s: str) -> bool:
    return bool(pat.search(s))

df["has_emoji"] = df["full_text"].apply(lambda s: has(EMOJI_RE, s))
df["has_emoticon"] = df["full_text"].apply(lambda s: has(EMOTICON_RE, s))

df["has_slang_abbrev"] = df["full_text"].apply(lambda s: has(SLANG_ABBREV_RE, s))
df["has_colloquial"] = df["full_text"].apply(lambda s: has(COLLOQUIAL_RE, s))
df["has_slang_any"] = df["has_slang_abbrev"] | df["has_colloquial"]

df["has_excl3"] = df["full_text"].apply(lambda s: has(EXCL_3_RE, s))
df["has_q3"] = df["full_text"].apply(lambda s: has(Q_3_RE, s))
df["has_allcaps"] = df["full_text"].apply(lambda s: has(ALLCAPS_WORD_RE, s))
df["has_elongated"] = df["full_text"].apply(lambda s: has(ELONGATED_RE, s))

df["any_informal"] = df["has_emoji"] | df["has_emoticon"] | df["has_slang_any"]
df["any_emphasis"] = df["has_excl3"] | df["has_q3"] | df["has_allcaps"] | df["has_elongated"]

headers = [
    "Group","N","Share of total (%)","Emoji (%)","Emoticon (%)","Slang markers (%)",
    "Any informal (%)","≥3 ! (%)","≥3 ? (%)","ALL CAPS (%)","Elongated (%)","Any emphasis (%)"
]

def summarize(sub: pd.DataFrame):
    n = len(sub)
    return [
        f"{n:,}",
        f"{(n/TOTAL_N*100):.2f}",
        f"{(sub['has_emoji'].mean()*100):.2f}",
        f"{(sub['has_emoticon'].mean()*100):.2f}",
        f"{(sub['has_slang_any'].mean()*100):.2f}",
        f"{(sub['any_informal'].mean()*100):.2f}",
        f"{(sub['has_excl3'].mean()*100):.2f}",
        f"{(sub['has_q3'].mean()*100):.2f}",
        f"{(sub['has_allcaps'].mean()*100):.2f}",
        f"{(sub['has_elongated'].mean()*100):.2f}",
        f"{(sub['any_emphasis'].mean()*100):.2f}",
    ]

rows = [
    ["Overall (cleaned dataset)"] + summarize(df),
    ["Adidas"] + summarize(df[df["brand"]=="Adidas"]),
    ["Nike"] + summarize(df[df["brand"]=="Nike"]),
]

table = pd.DataFrame(rows, columns=headers)

csv_path = OUT_DIR / "Review_Characteristics_Overview_Brand.csv"
table.to_csv(csv_path, index=False)

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT_FAMILY = "Aptos" if "Aptos" in available_fonts else "DejaVu Sans"

mpl.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": 10.5,
    "savefig.dpi": 300,
})

fig, ax = plt.subplots(figsize=(12.8, 2.8))
ax.axis("off")

col_widths = [0.22, 0.08, 0.10, 0.08, 0.09, 0.11, 0.10, 0.07, 0.07, 0.08, 0.08, 0.10]
col_widths = [w/sum(col_widths) for w in col_widths]

tbl = ax.table(
    cellText=table.values.tolist(),
    colLabels=headers,
    cellLoc="center",
    colLoc="center",
    colWidths=col_widths,
    loc="upper left"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)

for (r,c), cell in tbl.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_linewidth(0.0)
    cell.set_height(0.34)

for c in range(len(headers)):
    tbl[(0,c)].set_text_props(weight="bold")

for r in range(1, len(rows)+1):
    tbl[(r,0)].set_text_props(ha="left")
    tbl[(r,0)]._loc = "left"

ax.plot([0,1],[0.95,0.95], color="black", lw=1.6, transform=ax.transAxes, clip_on=False) 
ax.plot([0,1],[0.75,0.75], color="black", lw=1.1, transform=ax.transAxes, clip_on=False) 
ax.plot([0,1],[0.51,0.51], color="black", lw=0.9, transform=ax.transAxes, clip_on=False) 
ax.plot([0,1],[0.08,0.08], color="black", lw=1.6, transform=ax.transAxes, clip_on=False)

ax.text(
    0, 1.02,
    "Review characteristics by brand (informal language & emphasis markers)",
    transform=ax.transAxes, ha="left", va="bottom",
    fontsize=11.5, weight="bold"
)

png_path = OUT_DIR / "Review_Characteristics_Table_Brand.png"
fig.tight_layout()
fig.savefig(png_path, bbox_inches="tight")
plt.close(fig)

print("Saved:", png_path)
print("Saved:", csv_path)


In [ ]:
# Analyse für Kapitel 4.1.2: Human Coder Validation
# Genutzt zum Erstellen von Tabelle 7 (Cohen's Kappa, SieBERT vs. VADER vs. Human)

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, confusion_matrix)

path = Path.home() / 'Desktop' / 'Masterarbeit' / 'SA'

main = pd.read_csv(path / 'reviews_combined_results.csv')
luc = pd.read_excel(path / 'Luc_Human_Validation.xlsx')
sr = pd.read_excel(path / 'Human_Validation_SR.xlsx')

luc.columns = luc.columns.str.strip()
sr.columns = sr.columns.str.strip()
luc['Sentiment'] = luc['Sentiment'].str.strip().str.lower()
sr['Sentiment'] = sr['Sentiment'].str.strip().str.lower()

agree = (luc['Sentiment'] == sr['Sentiment']).sum()
total = len(luc)
percent_agreement = agree / total * 100
kappa_inter = cohen_kappa_score(luc['Sentiment'], sr['Sentiment'])

print("=" * 60)
print("INTER-CODER RELIABILITY")
print("=" * 60)
print(f"Total reviews:      {total}")
print(f"Agreements:         {agree}")
print(f"Disagreements:      {total - agree}")
print(f"Percent Agreement:  {percent_agreement:.2f}%")
print(f"Cohen's Kappa:      {kappa_inter:.4f}")

disagree_mask = luc['Sentiment'] != sr['Sentiment']
print(f"\nDisagreement details:")
for _, row in luc[disagree_mask].iterrows():
    sr_label = sr.loc[_, 'Sentiment']
    print(f"  Review #{row['No.']}: Luc={row['Sentiment']}, SR={sr_label}")

human = luc[['No.', 'Review Text']].copy()
human['Coder_Luc'] = luc['Sentiment']
human['Coder_SR'] = sr['Sentiment']

human['Human_Gold'] = np.where(
    human['Coder_Luc'] == human['Coder_SR'],
    human['Coder_Luc'],
    None
)
tiebreaker = {
    2: 'positive',    
    46: 'negative',  
    148: 'negative',  
    150: 'negative', 
    159: 'positive', 
    184: 'positive'  
}

for review_no, label in tiebreaker.items():
    human.loc[human['No.'] == review_no, 'Human_Gold'] = label

print(f"\nGold Standard distribution:")
print(human['Human_Gold'].value_counts())

main['Combined'] = (main['Title'].fillna('') + ' ' + main['Text'].fillna('')).str.strip()

merged = human.merge(main, left_on='Review Text', right_on='Combined', how='left')

mask_unmatched = merged['SieBERT_Sentiment'].isna()
if mask_unmatched.sum() > 0:
    print(f"\nUnmatched reviews: {mask_unmatched.sum()}. Attempting fuzzy match...")
    for idx in merged[mask_unmatched].index:
        review_norm = merged.loc[idx, 'Review Text'].replace('.', '').replace(',', '').strip()
        for _, mrow in main.iterrows():
            combined_norm = mrow['Combined'].replace('.', '').replace(',', '').strip()
            if review_norm[:50] in combined_norm or combined_norm[:50] in review_norm:
                for col in ['SieBERT_Sentiment', 'SieBERT_Confidence',
                            'VADER_Sentiment', 'VADER_Compound', 'Ground_Truth']:
                    merged.loc[idx, col] = mrow[col]
                print(f"  Fixed review #{merged.loc[idx, 'No.']}")
                break

print(f"Successfully matched: {merged['SieBERT_Sentiment'].notna().sum()}/{total}")

final = merged[['No.', 'Review Text', 'Coder_Luc', 'Coder_SR', 'Human_Gold',
                'SieBERT_Sentiment', 'SieBERT_Confidence',
                'VADER_Sentiment', 'VADER_Compound', 'Ground_Truth']].copy()

final.to_csv(path / 'Final_Human_Coder.csv', index=False)
print("\nFinal_Human_Coder.csv saved.")

y_true = final['Human_Gold']
y_siebert = final['SieBERT_Sentiment'].str.lower()
y_vader = final['VADER_Sentiment'].str.lower()


def compute_metrics(y_true, y_pred, model_name):
    """Compute and print all classification metrics."""
    acc = accuracy_score(y_true, y_pred)
    prec_pos = precision_score(y_true, y_pred, pos_label='positive')
    rec_pos = recall_score(y_true, y_pred, pos_label='positive')
    f1_pos = f1_score(y_true, y_pred, pos_label='positive')
    prec_neg = precision_score(y_true, y_pred, pos_label='negative')
    rec_neg = recall_score(y_true, y_pred, pos_label='negative')
    f1_neg = f1_score(y_true, y_pred, pos_label='negative')
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    macro_prec = precision_score(y_true, y_pred, average='macro')
    macro_rec = recall_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=['positive', 'negative'])

    print(f"\n{'=' * 60}")
    print(f"{model_name} vs Human Gold Standard (n={len(y_true)})")
    print(f"{'=' * 60}")
    print(f"Accuracy:           {acc:.4f}")
    print(f"Precision (Pos):    {prec_pos:.4f}")
    print(f"Recall (Pos):       {rec_pos:.4f}")
    print(f"F1 (Pos):           {f1_pos:.4f}")
    print(f"Precision (Neg):    {prec_neg:.4f}")
    print(f"Recall (Neg):       {rec_neg:.4f}")
    print(f"F1 (Neg):           {f1_neg:.4f}")
    print(f"Macro-Precision:    {macro_prec:.4f}")
    print(f"Macro-Recall:       {macro_rec:.4f}")
    print(f"Macro-F1:           {macro_f1:.4f}")
    print(f"Cohen's Kappa:      {kappa:.4f}")
    print(f"\nConfusion Matrix (rows=true, cols=predicted):")
    print(f"              Pred_Pos  Pred_Neg")
    print(f"True_Pos      {cm[0, 0]:>6}    {cm[0, 1]:>6}")
    print(f"True_Neg      {cm[1, 0]:>6}    {cm[1, 1]:>6}")

    return {
        'Accuracy': acc, 'Precision (Positive)': prec_pos,
        'Recall (Positive)': rec_pos, 'F1-Score (Positive)': f1_pos,
        'Precision (Negative)': prec_neg, 'Recall (Negative)': rec_neg,
        'F1-Score (Negative)': f1_neg,
        'Macro-Precision': macro_prec, 'Macro-Recall': macro_rec,
        'Macro-F1': macro_f1, "Cohen's Kappa": kappa
    }


siebert_metrics = compute_metrics(y_true, y_siebert, "SieBERT")
vader_metrics = compute_metrics(y_true, y_vader, "VADER")

summary = pd.DataFrame({
    'Metric': list(siebert_metrics.keys()),
    'SieBERT': [f"{v:.4f}" for v in siebert_metrics.values()],
    'VADER': [f"{v:.4f}" for v in vader_metrics.values()]
})

print(f"\n{'=' * 60}")
print("SUMMARY TABLE")
print(f"{'=' * 60}")
print(summary.to_string(index=False))

In [ ]:

# Genutzt zum Erstellen von Fig. 10: Review Length by Brand and Product Type (Boxplot)


import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import to_rgb

DATA = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "Data" / "All_Reviews_Organic.csv"
OUT  = Path.home() / "Desktop" / "Masterarbeit" / "Figures" / "Methodology"
OUT.mkdir(parents=True, exist_ok=True)

UNI_GREEN = "#009260"
def mix(hex_color, t):
    r,g,b = to_rgb(hex_color)
    return (r+(1-r)*t, g+(1-g)*t, b+(1-b)*t)

df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.strip()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["Rating"].between(1,5)].copy()
df["word_count"] = df["Text"].fillna("").apply(lambda s: len(s.split()))

groups = [
    ("Adidas Conventional", df[(df["Brand"]=="Adidas") & (df["Sustainability"]=="Conventional")]),
    ("Adidas Sustainable",  df[(df["Brand"]=="Adidas") & (df["Sustainability"]=="Sustainable")]),
    ("Nike Conventional",   df[(df["Brand"]=="Nike")   & (df["Sustainability"]=="Conventional")]),
    ("Nike Sustainable",    df[(df["Brand"]=="Nike")    & (df["Sustainability"]=="Sustainable")]),
]

p99 = int(np.ceil(df["word_count"].quantile(0.99)))

fig, ax = plt.subplots(figsize=(9, 4.2))

bp = ax.boxplot(
    [g[1]["word_count"] for g in groups],
    vert=False,
    labels=[f"{g[0]} (n={len(g[1]):,})" for g in groups],
    patch_artist=True,
    showmeans=True,
    meanprops=dict(marker="o", markerfacecolor="black", markeredgecolor="black", markersize=5),
    medianprops=dict(color="black", linewidth=1.2),
    flierprops=dict(marker="", linewidth=0), 
)

colors = [mix(UNI_GREEN, t) for t in [0.55, 0.35, 0.55, 0.35]]
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_edgecolor("black")

ax.set_xlim(0, p99 + 5)
ax.set_xlabel("Word count (review body)")
ax.set_title("Review Length by Brand and Product Type", fontsize=14)

ax.annotate(f"x-axis truncated at p99 ≈ {p99} words",
            xy=(1, 0), xycoords="axes fraction", ha="right", va="top",
            fontsize=8, fontstyle="italic", color="grey")

plt.tight_layout()
fig.savefig(OUT / "fig10_review_length_boxplot.png", dpi=300)
fig.savefig(OUT / "fig10_review_length_boxplot.pdf")
plt.show()
print("Saved Fig. 10")

In [ ]:
# Genutzt zum Erstellen von Fig. 11: Star Rating Distribution by Review Length


SHADES = [mix(UNI_GREEN, t) for t in [0.82, 0.72, 0.60, 0.35, 0.00]]

short = df[df["word_count"] < 14].copy()
long_ = df[df["word_count"] > 25].copy()

def star_pct(sub):
    stars = sub["Rating"].round().astype(int)
    counts = stars.value_counts().reindex([1,2,3,4,5], fill_value=0)
    return (counts / len(sub) * 100)

pct_short = star_pct(short)
pct_long  = star_pct(long_)

fig, ax = plt.subplots(figsize=(9, 3.2))

labels = [
    f"Long (>25 words) (n={len(long_):,})",
    f"Short (<14 words) (n={len(short):,})",
]
data = [pct_long.values, pct_short.values]

y = np.arange(len(labels))
lefts = [np.zeros(len(labels)) for _ in range(5)]

for star_idx in range(5):
    vals = [d[star_idx] for d in data]
    left = [sum(d[:star_idx]) for d in data]
    bars = ax.barh(y, vals, left=left, height=0.5, color=SHADES[star_idx],
                   label=f"{star_idx+1}★", edgecolor="white", linewidth=0.5)

ax.set_yticks(y, labels)
ax.set_xlim(0, 100)
ax.set_xlabel("Share of reviews")
ax.set_title("Star Rating Distribution by Review Length", fontsize=14)
ax.legend(loc="lower right", ncol=5, fontsize=9, frameon=False)

# Annotate 5-star share
ax.text(98, 1, f"5★: {pct_short[5]:.1f}%", va="center", ha="right", fontsize=10, color="white", fontweight="bold")
ax.text(98, 0, f"5★: {pct_long[5]:.1f}%", va="center", ha="right", fontsize=10, color="white", fontweight="bold")

plt.tight_layout()
fig.savefig(OUT / "fig11_star_rating_by_review_length.png", dpi=300)
fig.savefig(OUT / "fig11_star_rating_by_review_length.pdf")
plt.show()
print("Saved Fig. 11")